# P2 — Crypto Inventory & Migration Planner

Phase 1 reproducible implementation.

## 1. Select the problem statement
P2 combines a classical cryptographic inventory and a measured A/B quantum processor benchmark.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))
import pandas as pd
import matplotlib.pyplot as plt
from src.problem import score_inventory
from src.processors import load_processor
from src.routing import build_grover_circuit
from src.metrics import run_processor_benchmark, compare_metrics, compare_three_metrics
from src.visualization import make_figures
from src.discovery import discover_directory


## 2. Understand the inputs
The raw inventory has the required five columns. Processor JSON files contain architecture, basis-gate, port, and placeholder noise definitions.

In [ ]:
inventory = pd.read_csv(ROOT/'data/raw/inventory.csv')
processor_a = load_processor(ROOT/'processors/processor_A.json')
processor_b = load_processor(ROOT/'processors/processor_B.json')
processor_d = load_processor(ROOT/'processors/processor_D.json')
display(inventory)
print(processor_a)
print(processor_b)
print(processor_d)

## 3. Build the inventory scorer

In [ ]:
scored = score_inventory(ROOT/'data/raw/inventory.csv', ROOT/'results/tables')
display(scored)

## 4. Authorized local discovery demonstration
The discovery module scans only the explicitly supplied synthetic fixture directory. It performs no network scanning and returns source-file evidence for human review. CSV import remains the fallback for inventories supplied by the Challenge Kit.

In [ ]:
discovered = discover_directory(ROOT/'data/raw/authorized_test_env')
discovered.to_csv(ROOT/'results/tables/discovered_assets.csv', index=False)
discovered

## 4. Build the logical Grover circuit

In [ ]:
logical = build_grover_circuit(marked='101', iterations=2, measure=True)
display(logical.draw('mpl'))

## 5. Processor A — transpile, execute, and measure

In [ ]:
row_a, compiled_a, baseline_a = run_processor_benchmark(logical, processor_a)
row_a

## 6. Processor B — same circuit and same settings

In [ ]:
row_b, compiled_b, baseline_b = run_processor_benchmark(logical, processor_b)
row_b

## 7. Supplementary Processor D — same logical circuit and settings

Processor D is a participant-defined 12-qubit research model. This additional run uses the same marked-state Grover circuit, shots, seed, basis-gate policy, and model noise assumptions as A/B. It supplements rather than replaces the required A/B comparison.

In [ ]:
row_d, compiled_d, baseline_d = run_processor_benchmark(logical, processor_d)
row_d

## 8. Required A vs B table and supplementary A/B/D comparison

In [ ]:
comparison = pd.DataFrame(compare_metrics(row_a, row_b))
comparison.to_csv(ROOT/'results/tables/AB_comparison.csv', index=False)
comparison.to_csv(ROOT/'results/tables/final_comparison.csv', index=False)
pd.DataFrame([row_a]).to_csv(ROOT/'results/tables/processor_A_metrics.csv', index=False)
pd.DataFrame([row_b]).to_csv(ROOT/'results/tables/processor_B_metrics.csv', index=False)
pd.DataFrame([row_d]).to_csv(ROOT/'results/tables/processor_D_metrics.csv', index=False)
abd_comparison = pd.DataFrame(compare_three_metrics(row_a, row_b, row_d))
abd_comparison.to_csv(ROOT/'results/tables/ABD_comparison.csv', index=False)
display(comparison)
display(abd_comparison)

## 8. Explain architecture effects
The coupling map determines how easily logical two-qubit interactions can be placed. A more connected map is not automatically better: the measured success probability also depends on the compiled gate sequence and the shared noise model. The A/B result table is the evidence used below.

In [ ]:
make_figures(
    {'processor_A':processor_a, 'processor_B':processor_b, 'processor_D':processor_d},
    logical,
    {'processor_A':compiled_a, 'processor_B':compiled_b, 'processor_D':compiled_d},
    [row_a,row_b,row_d],
    ROOT/'results/figures'
)
print('Figures written.')

## 9. Figures

In [ ]:
from IPython.display import Image, display
for name in ['processor_A_graph.png','processor_B_graph.png','processor_D_graph.png','logical_vs_transpiled.png','AB_performance.png','architecture_tradeoffs.png']:
    print(name); display(Image(filename=str(ROOT/'results/figures'/name)))

## 10. Reproducibility notes

In [ ]:
# Generate report and data-driven README results section.
report = "# P2 Report — Crypto Inventory & Migration Planner\n\n"
report += "## What changed?\nThe inventory was scored with the specified lookup tables, and each asset received a migration recommendation and phase. The same marked-state Grover circuit was compiled and executed on the required participant-defined A/B research models. Processor D was additionally benchmarked as a supplementary 12-qubit participant-defined research model. A has 5 qubits; B has 9 qubits; D has 12 qubits. The coupling maps are research designs, and noise values are controlled simulator assumptions, not hardware calibration measurements.\n\n"
report += "## Benchmark settings\nThe benchmark follows the guide settings: 4096 shots, optimization level 1, seed 42, one-qubit depolarizing error 0.001, CX depolarizing error 0.005, and readout error 0.01. These are illustrative simulator assumptions, not official calibration data. The measurements reported below are generated by this notebook run using these settings.\n\n"
report += "## PS-P2 public-repository scanner evaluation\nThe local synthetic fixture is a demonstration only. Run `python scripts/evaluate_public_repo.py --repo psf/requests --ref main` to scan a public repository and record scanner execution time. Precision and recall remain pending until a complete manual ground-truth audit is supplied; this report does not claim unmeasured values.\n\n"
report += f"## Which architectural property changed?\nProcessor A uses coupling edges {processor_a['coupling_map']} with graph properties {processor_a['graph_properties']}; Processor B uses coupling edges {processor_b['coupling_map']} with graph properties {processor_b['graph_properties']}; Processor D uses coupling edges {processor_d['coupling_map']} with graph properties {processor_d['graph_properties']}. B and D are participant-defined sparse research graphs with different graph structure; all topology, basis-gate, and noise details are explicit simulator assumptions rather than official calibration data.\n\n"
report += f"## Did transpilation add routing?\nUsing the all-to-all baseline, Processor A compiled to {row_a['two_qubit_gates']} counted two-qubit gates versus {row_a['baseline_two_qubit_gates']} baseline, for an estimated SWAP overhead of {row_a['swap_estimate']:.6f}; Processor B compiled to {row_b['two_qubit_gates']} versus {row_b['baseline_two_qubit_gates']} baseline, for an estimated SWAP overhead of {row_b['swap_estimate']:.6f}; supplementary Processor D compiled to {row_d['two_qubit_gates']} versus {row_d['baseline_two_qubit_gates']} baseline, for an estimated SWAP overhead of {row_d['swap_estimate']:.6f}. The direct measured SWAP counts are {row_a['measured_swap_count']}, {row_b['measured_swap_count']}, and {row_d['measured_swap_count']} for A, B, and D respectively. SWAP-equivalents are estimates, not direct SWAP instruction counts.\n\n"
report += f"## How many extra operations?\nThe total compiled operation counts were {row_a['total_ops']} for Processor A, {row_b['total_ops']} for Processor B, and {row_d['total_ops']} for supplementary Processor D; extra two-qubit counts versus baseline were {row_a['extra_two_qubit_gates']}, {row_b['extra_two_qubit_gates']}, and {row_d['extra_two_qubit_gates']}.\n\n"
report += f"## Did the problem metric change?\nThe measured success probabilities for state `101` were {row_a['success_probability']:.6f} ± {row_a['std_error']:.6f} for Processor A, {row_b['success_probability']:.6f} ± {row_b['std_error']:.6f} for Processor B, and {row_d['success_probability']:.6f} ± {row_d['std_error']:.6f} for supplementary Processor D. These measurements apply only to this three-qubit Grover workload and the participant-defined simulator models. Ideal reference probabilities were {row_a['ideal_success_probability']:.6f}, {row_b['ideal_success_probability']:.6f}, and {row_d['ideal_success_probability']:.6f}.\n\n"
report += "## Could gate-set/noise differences also contribute?\nYes. This run keeps basis gates and the default placeholder noise model identical, but transpilation can produce different gate counts and layouts. If official processor data changes basis gates, ports, calibration, or noise, those factors can contribute and must be reported separately.\n\n"
report += "## Why both components belong in this project?\nThe inventory planner answers which cryptographic assets an organization should migrate first. The Grover experiment answers the separate Phase 1 architecture question: how processor topology changes execution of the same quantum solution. It is not a measurement of the cost of attacking real RSA or ECC implementations.\n\n"
report += "## What remains uncertain?\nThe guide intentionally does not provide official Processor A/B JSON files. This submission therefore defines and documents its own reproducible A/B graphs while following the specified qubit counts, basis gates, and benchmark settings. The graphs are research models, not claims about hidden official edge lists or calibrated hardware. Runtime is environment-dependent. The SWAP value is an estimate rather than a direct decomposition count.\n\n"
report += "## Limitations\nA toy three-qubit Grover run does not estimate the cost of attacking real RSA/ECC keys. It is an educational architecture comparison, not a cryptanalytic forecast.\n"
from src.architecture_stress import run_architecture_stress_test
stress = run_architecture_stress_test(
    ROOT/'processors/processor_A.json',
    ROOT/'processors/processor_B.json',
)
stress.to_csv(ROOT/'results/tables/architecture_stress_test.csv', index=False)
report += "## Supplementary Processor D benchmark\n\n"
report += "Processor D is a participant-defined 12-qubit research model, not official calibrated hardware. The same three-qubit Grover circuit and simulator settings are used for a supplemental comparison. The required A/B table remains separate.\n\n"
report += "| Metric | Processor A | Processor B | Processor D | B - A | D - A | D - B |\n|---|---:|---:|---:|---:|---:|---:|\n"
for label, field, fmt in [
    ("Success probability for 101", "success_probability", ".6f"),
    ("Standard error", "std_error", ".6f"),
    ("Compiled depth", "depth", "d"),
    ("Counted two-qubit gates", "two_qubit_gates", "d"),
    ("Estimated SWAP-equivalents", "swap_estimate", ".3f"),
    ("Measured SWAP instructions", "measured_swap_count", "d"),
    ("Physical qubits used", "physical_qubits_used", "d"),
    ("Total compiled operations", "total_ops", "d"),
    ("Runtime seconds", "runtime_seconds", ".6f"),
]:
    av, bv, dv = row_a[field], row_b[field], row_d[field]
    report += f"| {label} | {format(av, fmt)} | {format(bv, fmt)} | {format(dv, fmt)} | {format(bv-av, fmt)} | {format(dv-av, fmt)} | {format(dv-bv, fmt)} |\n"
report += "\nProcessor D is synthetic and exploratory. Runtime is environment-dependent; this is not an official Processor C/D Bell-correlation experiment.\n\n"
report += "## Supplementary topology stress test\n\n"
report += "The required three-qubit Grover benchmark can produce identical A/B metrics even when the participant-defined coupling maps differ. Supplementary synthetic workloads probe topology sensitivity without replacing the required benchmark.\n\n"
report += "| Workload | A depth | B depth | A two-qubit gates | B two-qubit gates | A estimated SWAP-equivalents | B estimated SWAP-equivalents |\n"
report += "|---|---:|---:|---:|---:|---:|---:|\n"
for workload_name in ("star", "chain", "ring"):
    a_stress = stress.loc[(stress["workload"] == workload_name) & (stress["processor"] == "Processor A")].iloc[0]
    b_stress = stress.loc[(stress["workload"] == workload_name) & (stress["processor"] == "Processor B")].iloc[0]
    report += f"| {workload_name} | {int(a_stress['compiled_depth'])} | {int(b_stress['compiled_depth'])} | {int(a_stress['compiled_two_qubit_gates'])} | {int(b_stress['compiled_two_qubit_gates'])} | {a_stress['estimated_swap_equivalents']:.3f} | {b_stress['estimated_swap_equivalents']:.3f} |\n"
report += "\nEstimated SWAP-equivalents are extra two-qubit gates over an all-to-all baseline divided by three; this is a proxy, not a direct SWAP count. Results are simulator/compiler metrics for participant-defined models, not official hardware results or cryptographic attack estimates.\n\n"
(ROOT/'report.md').write_text(report, encoding='utf-8')
readme = (ROOT/'README.md').read_text(encoding='utf-8')
interpretation = (
    "Processor B improved both measured success probability and compiled operation count for this run."
    if row_b['success_probability'] > row_a['success_probability'] and row_b['total_ops'] < row_a['total_ops']
    else "Processor A and Processor B produced equal measured success probability and operation count for this workload."
    if row_b['success_probability'] == row_a['success_probability'] and row_b['total_ops'] == row_a['total_ops']
    else "The measured A/B differences are mixed; interpret each metric independently."
)
metrics = [
    ("Success probability for 101", "success_probability", ".6f"),
    ("Standard error", "std_error", ".6f"),
    ("Compiled depth", "depth", "d"),
    ("Counted two-qubit gates", "two_qubit_gates", "d"),
    ("Estimated SWAP-equivalents", "swap_estimate", ".3f"),
    ("Measured SWAP instructions", "measured_swap_count", "d"),
    ("Physical qubits used", "physical_qubits_used", "d"),
    ("Total compiled operations", "total_ops", "d"),
    ("Runtime seconds", "runtime_seconds", ".6f"),
]
result_section = "## Verified run results\\n\\nGenerated by the executed notebook using the pinned benchmark settings.\\n\\n"
result_section += "| Metric | Processor A | Processor B | B - A |\\n|---|---:|---:|---:|\\n"
for label, field, fmt in metrics:
    av, bv = row_a[field], row_b[field]
    result_section += f"| {label} | {format(av, fmt)} | {format(bv, fmt)} | {format(bv-av, fmt)} |\\n"
result_section += f"\\n**Interpretation:** {interpretation} This conclusion applies only to this workload and simulator model; it is not a universal claim about processor superiority.\\n\\n"
result_section += "### Supplementary Processor D benchmark\\n\\n"
result_section += "Processor D is a participant-defined 12-qubit research model run on the same logical three-qubit Grover circuit and simulator settings. The required A/B comparison above remains primary. D is not an official hardware result or the separate offline C/D Bell-correlation study.\\n\\n"
result_section += "| Metric | Processor A | Processor B | Processor D | B - A | D - A | D - B |\\n|---|---:|---:|---:|---:|---:|---:|\\n"
for label, field, fmt in metrics:
    av, bv, dv = row_a[field], row_b[field], row_d[field]
    result_section += f"| {label} | {format(av, fmt)} | {format(bv, fmt)} | {format(dv, fmt)} | {format(bv-av, fmt)} | {format(dv-av, fmt)} | {format(dv-bv, fmt)} |\\n"
result_section += "\\nCSV evidence: [processor_D_metrics.csv](results/tables/processor_D_metrics.csv) and [ABD_comparison.csv](results/tables/ABD_comparison.csv).\\n\\n"
result_section += "### Supplementary topology stress test\\n\\n"
result_section += "The star, chain, and ring compiler stress tests compare A and B and complement but do not replace the required Grover benchmark.\\n\\n"
result_section += "| Workload | A depth | B depth | A two-qubit gates | B two-qubit gates | A estimated SWAP-equivalents | B estimated SWAP-equivalents |\\n|---|---:|---:|---:|---:|---:|---:|\\n"
for workload_name in ("star", "chain", "ring"):
    a_stress = stress.loc[(stress["workload"] == workload_name) & (stress["processor"] == "Processor A")].iloc[0]
    b_stress = stress.loc[(stress["workload"] == workload_name) & (stress["processor"] == "Processor B")].iloc[0]
    result_section += f"| {workload_name} | {int(a_stress['compiled_depth'])} | {int(b_stress['compiled_depth'])} | {int(a_stress['compiled_two_qubit_gates'])} | {int(b_stress['compiled_two_qubit_gates'])} | {a_stress['estimated_swap_equivalents']:.3f} | {b_stress['estimated_swap_equivalents']:.3f} |\\n"
result_section += "\\nEstimated SWAP-equivalents are a proxy, not a direct SWAP count. Full analysis: [report.md](report.md); machine-readable evidence: [results/tables](results/tables).\\n\\n---\\n\\n"
start, end = "## Latest committed benchmark snapshot", "## Quick start"
if start in readme and end in readme:
    before = readme.split(start, 1)[0]
    after = end + readme.split(end, 1)[1]
    readme = before + result_section + after
(ROOT/'README.md').write_text(readme, encoding='utf-8')
